# 03 - Model Loading and LoRA Setup (Google Colab)
Load Phi base model, configure 4-bit quantization, attach LoRA adapters, and inspect trainable parameters.

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import yaml
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

In [ ]:
# Load configurations
with open("configs/training_config.yaml", "r") as f:
    train_cfg = yaml.safe_load(f)

with open("configs/lora_config.yaml", "r") as f:
    lora_cfg = yaml.safe_load(f)

print("Training Config:")
print(f"  Model: {train_cfg['model_name']}")
print(f"  Fallback: {train_cfg['fallback_model']}")
print(f"  4-bit: {train_cfg['use_4bit']}")
print(f"  Max seq length: {train_cfg['max_seq_length']}")

print(f"\nLoRA Config:")
print(f"  r: {lora_cfg['r']}")
print(f"  alpha: {lora_cfg['lora_alpha']}")
print(f"  dropout: {lora_cfg['lora_dropout']}")
print(f"  target modules: {lora_cfg['target_modules']}")
print(f"  bias: {lora_cfg['bias']}")

In [ ]:
# GPU check
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    use_4bit = train_cfg["use_4bit"]
else:
    print("No GPU available - disabling quantization")
    use_4bit = False

In [ ]:
# Step 1: Configure quantization
bnb_config = None

if use_4bit:
    compute_dtype = getattr(torch, train_cfg.get("bnb_4bit_compute_dtype", "float16"))
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=train_cfg.get("bnb_4bit_quant_type", "nf4"),
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=train_cfg.get("use_double_quant", True),
    )
    print("4-bit quantization configured (NF4 + double quant)")
else:
    print("Quantization disabled")

In [ ]:
# Step 2: Load tokenizer
model_name = train_cfg["model_name"]
print(f"Loading tokenizer: {model_name}")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    print(f"Tokenizer loaded: {model_name}")
except Exception as e:
    print(f"Failed to load primary model tokenizer: {e}")
    model_name = train_cfg["fallback_model"]
    print(f"Loading fallback tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

In [ ]:
# Step 3: Load base model
print(f"Loading base model: {model_name}")
print("This may take a few minutes...")

model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "device_map": "auto",
}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

print(f"\nModel loaded: {model_name}")
print(f"Model type: {type(model).__name__}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Step 4: Enable gradient checkpointing
if train_cfg.get("gradient_checkpointing", True):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

# Prepare for quantized training
if bnb_config is not None:
    model = prepare_model_for_kbit_training(model)
    print("Model prepared for k-bit training")

In [ ]:
# Step 5: Configure LoRA adapters
lora_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Bias: {lora_config.bias}")
print(f"  Task type: {lora_config.task_type}")

In [ ]:
# Step 6: Apply LoRA and inspect
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params, total_params = model.get_nb_trainable_parameters()
print(f"\nParameter Summary:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")
print(f"  Memory savings: ~{(1 - trainable_params/total_params)*100:.1f}% fewer params to train")

In [ ]:
# Step 7: Print model architecture with LoRA modules highlighted
print("LoRA-adapted modules:")
for name, module in model.named_modules():
    if "lora" in name.lower():
        print(f"  {name}: {type(module).__name__}")

In [ ]:
# Quick sanity check - run a forward pass
test_input = tokenizer("### Instruction:\nEvaluate the resume", return_tensors="pt")
test_input = {k: v.to(model.device) for k, v in test_input.items()}

with torch.no_grad():
    output = model(**test_input)

print(f"Forward pass successful!")
print(f"Output logits shape: {output.logits.shape}")
print(f"\nModel is ready for fine-tuning!")